# RADAR Seatbelt + Mobile Detection — YOLOv11m Training

**Model**: YOLOv11m (medium) — fine-tuned from COCO pretrained weights  
**Task**: Detect seatbelt usage and mobile phone usage from front-facing Egyptian radar camera images  
**Classes**: 5 (person-noseatbelt, person-seatbelt, seatbelt, windshield, mobile)  
**Dataset**: 6,424 images (all 640x640) — merged from v3 base + v2 mobile labels + v5 unique images  
**Target**: Kaggle T4 GPU, batch=32, epochs=100

## 1. Setup

In [ ]:
!pip install -U ultralytics -q

import os
import glob
import random
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from collections import Counter
from PIL import Image

from ultralytics import YOLO

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Ultralytics version: {__import__('ultralytics').__version__}")

In [ ]:
# Check GPU
!nvidia-smi

## 2. Dataset Configuration

In [ ]:
DATASET_DIR = "/kaggle/input/datasets/marwanmesbah19/theoneandonlybela/seatbelt_merged"
WORK_DIR = "/kaggle/working"

# Verify dataset structure
for split in ["train", "valid", "test"]:
    img_dir = os.path.join(DATASET_DIR, split, "images")
    lbl_dir = os.path.join(DATASET_DIR, split, "labels")
    n_img = len(glob.glob(os.path.join(img_dir, "*.*"))) if os.path.exists(img_dir) else 0
    n_lbl = len(glob.glob(os.path.join(lbl_dir, "*.txt"))) if os.path.exists(lbl_dir) else 0
    print(f"{split:>5}: {n_img:>5} images, {n_lbl:>5} labels")

In [ ]:
# Class definitions
CLASS_NAMES = [
    'person-noseatbelt',  # 0
    'person-seatbelt',    # 1
    'seatbelt',           # 2
    'windshield',         # 3
    'mobile',             # 4
]
CLASS_COLORS_HEX = ['#F44336', '#4CAF50', '#2196F3', '#FFC107', '#9C27B0']
NUM_CLASSES = len(CLASS_NAMES)
print(f"Number of classes: {NUM_CLASSES}")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {i}: {name}")

In [ ]:
# Write data.yaml for training (absolute Kaggle paths)
data_yaml = {
    'path': DATASET_DIR,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': NUM_CLASSES,
    'names': {i: name for i, name in enumerate(CLASS_NAMES)}
}

yaml_path = os.path.join(WORK_DIR, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"Written: {yaml_path}")
print(yaml.dump(data_yaml, default_flow_style=False))

## 3. Dataset Analysis & Visualization

In [ ]:
# Count instances per class across all splits
def count_classes(dataset_dir, class_names):
    counts = {"train": Counter(), "valid": Counter(), "test": Counter()}
    for split in ["train", "valid", "test"]:
        lbl_dir = os.path.join(dataset_dir, split, "labels")
        if not os.path.exists(lbl_dir):
            continue
        for lbl_file in glob.glob(os.path.join(lbl_dir, "*.txt")):
            with open(lbl_file) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cid = int(parts[0])
                        if cid < len(class_names):
                            counts[split][class_names[cid]] += 1
    return counts

class_counts = count_classes(DATASET_DIR, CLASS_NAMES)

In [ ]:
# Plot: Total instances per class
fig, ax = plt.subplots(figsize=(12, 6))
total_counts = {k: sum(class_counts[s][k] for s in class_counts) for k in CLASS_NAMES}
colors = ['#F44336', '#4CAF50', '#2196F3', '#FFC107', '#9C27B0']
bars = ax.bar(CLASS_NAMES, [total_counts[k] for k in CLASS_NAMES], color=colors)
ax.set_xlabel('Class', fontsize=12)
ax.set_ylabel('Instance Count', fontsize=12)
ax.set_title('Total Instances per Class (All Splits)', fontsize=14)
ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')

# Add value labels on bars
for bar, count in zip(bars, [total_counts[k] for k in CLASS_NAMES]):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
            f'{count:,}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'class_counts_total.png'), dpi=150)
plt.show()

print(f"Total instances: {sum(total_counts.values()):,}")
print(f"Most common: {max(total_counts, key=total_counts.get)} ({max(total_counts.values()):,})")
print(f"Least common: {min(total_counts, key=total_counts.get)} ({min(total_counts.values()):,})")
print(f"Imbalance ratio: {max(total_counts.values()) / max(1, min(total_counts.values())):.1f}x")

In [ ]:
# Plot: Per-split class distribution (grouped bar)
fig, ax = plt.subplots(figsize=(12, 6))
train_vals = [class_counts['train'].get(k, 0) for k in CLASS_NAMES]
val_vals = [class_counts['valid'].get(k, 0) for k in CLASS_NAMES]
test_vals = [class_counts['test'].get(k, 0) for k in CLASS_NAMES]

x = np.arange(len(CLASS_NAMES))
w = 0.25
ax.bar(x - w, train_vals, w, label='Train', color='#4CAF50')
ax.bar(x, val_vals, w, label='Valid', color='#FF9800')
ax.bar(x + w, test_vals, w, label='Test', color='#F44336')
ax.set_xlabel('Class', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Class Distribution per Split', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'class_counts_per_split.png'), dpi=150)
plt.show()

In [ ]:
# Plot: Split sizes pie chart
split_totals = {s: sum(class_counts[s].values()) for s in class_counts}
split_imgs = {}
for s in ["train", "valid", "test"]:
    img_dir = os.path.join(DATASET_DIR, s, "images")
    split_imgs[s] = len(glob.glob(os.path.join(img_dir, "*.*"))) if os.path.exists(img_dir) else 0

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# By instances
axes[0].pie(split_totals.values(), labels=[f"{s.upper()}\n({v:,})" for s, v in split_totals.items()],
        autopct='%1.1f%%', colors=['#4CAF50', '#FF9800', '#F44336'], textprops={'fontsize': 13})
axes[0].set_title('Label Instances per Split', fontsize=14)

# By images
axes[1].pie(split_imgs.values(), labels=[f"{s.upper()}\n({v:,})" for s, v in split_imgs.items()],
        autopct='%1.1f%%', colors=['#4CAF50', '#FF9800', '#F44336'], textprops={'fontsize': 13})
axes[1].set_title('Images per Split', fontsize=14)

plt.suptitle('Dataset Split Distribution', fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'split_pie.png'), dpi=150)
plt.show()

In [ ]:
# Analyze image dimensions and bounding box sizes
img_sizes = []
bbox_sizes = []
bbox_aspect_ratios = []
instances_per_img = []

for split in ["train", "valid", "test"]:
    img_dir = os.path.join(DATASET_DIR, split, "images")
    lbl_dir = os.path.join(DATASET_DIR, split, "labels")
    if not os.path.exists(img_dir):
        continue
    for img_path in glob.glob(os.path.join(img_dir, "*.*"))[:500]:
        img = Image.open(img_path)
        img_sizes.append(img.size)
        lbl_path = os.path.join(lbl_dir, Path(img_path).stem + ".txt")
        if not os.path.exists(lbl_path):
            continue
        count = 0
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    bw, bh = float(parts[3]), float(parts[4])
                    bbox_sizes.append(bw * bh)
                    bbox_aspect_ratios.append(bw / max(bh, 0.001))
                    count += 1
        instances_per_img.append(count)

print(f"Analyzed {len(img_sizes)} images")
print(f"Unique image sizes: {set(img_sizes)}")

In [ ]:
# Plot: Image dimensions distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
widths = [s[0] for s in img_sizes]
heights = [s[1] for s in img_sizes]
axes[0].hist(widths, bins=30, color='#2196F3', alpha=0.7, edgecolor='black')
axes[0].set_title('Image Width Distribution')
axes[0].set_xlabel('Width (px)')
axes[0].axvline(np.mean(widths), color='red', linestyle='--', label=f'Mean: {np.mean(widths):.0f}')
axes[0].legend()

axes[1].hist(heights, bins=30, color='#FF9800', alpha=0.7, edgecolor='black')
axes[1].set_title('Image Height Distribution')
axes[1].set_xlabel('Height (px)')
axes[1].axvline(np.mean(heights), color='red', linestyle='--', label=f'Mean: {np.mean(heights):.0f}')
axes[1].legend()

plt.suptitle('Image Dimensions', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'image_dimensions.png'), dpi=150)
plt.show()

In [ ]:
# Plot: Bounding box sizes and aspect ratios
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(bbox_sizes, bins=50, color='#9C27B0', alpha=0.7, edgecolor='black')
axes[0].set_title('Bounding Box Area (normalized)')
axes[0].set_xlabel('Area (w*h)')
axes[0].axvline(np.mean(bbox_sizes), color='red', linestyle='--', label=f'Mean: {np.mean(bbox_sizes):.4f}')
axes[0].legend()

axes[1].hist(bbox_aspect_ratios, bins=50, color='#009688', alpha=0.7, edgecolor='black')
axes[1].set_title('Bounding Box Aspect Ratio (w/h)')
axes[1].set_xlabel('Aspect Ratio')
axes[1].axvline(np.mean(bbox_aspect_ratios), color='red', linestyle='--', label=f'Mean: {np.mean(bbox_aspect_ratios):.2f}')
axes[1].legend()

axes[2].hist(instances_per_img, bins=range(1, max(instances_per_img)+2),
             color='#FF5722', alpha=0.7, edgecolor='black', align='left')
axes[2].set_title('Instances per Image')
axes[2].set_xlabel('Number of Instances')
axes[2].set_ylabel('Count')
axes[2].axvline(np.mean(instances_per_img), color='red', linestyle='--',
                label=f'Mean: {np.mean(instances_per_img):.1f}')
axes[2].legend()

plt.suptitle('Bounding Box Analysis', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'bbox_analysis.png'), dpi=150)
plt.show()

## 4. Sample Images with Annotations

In [ ]:
def plot_sample_images(dataset_dir, split, class_names, class_colors, n_samples=8, seed=42):
    """Show sample images with YOLO bounding boxes and class labels."""
    img_dir = os.path.join(dataset_dir, split, "images")
    lbl_dir = os.path.join(dataset_dir, split, "labels")
    if not os.path.exists(img_dir):
        print(f"{split} split not found")
        return

    img_files = glob.glob(os.path.join(img_dir, "*.*"))
    random.seed(seed)
    samples = random.sample(img_files, min(n_samples, len(img_files)))

    cols = 4
    rows = (len(samples) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(20, 5 * rows))
    axes = axes.flatten() if rows > 1 else [axes] if cols == 1 else axes.flatten()

    for idx, (ax, img_path) in enumerate(zip(axes, samples)):
        img = Image.open(img_path)
        ax.imshow(np.array(img))

        lbl_path = os.path.join(lbl_dir, Path(img_path).stem + ".txt")
        if os.path.exists(lbl_path):
            iw, ih = img.size
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    cid, xc, yc, bw, bh = int(parts[0]), *[float(p) for p in parts[1:6]]
                    x1 = (xc - bw / 2) * iw
                    y1 = (yc - bh / 2) * ih
                    w = bw * iw
                    h = bh * ih
                    color = class_colors[cid] if cid < len(class_colors) else 'red'
                    rect = patches.Rectangle((x1, y1), w, h, linewidth=2,
                                            edgecolor=color, facecolor='none')
                    ax.add_patch(rect)
                    label = class_names[cid] if cid < len(class_names) else str(cid)
                    ax.text(x1, y1 - 3, label, fontsize=8, color='white',
                           bbox=dict(facecolor=color, alpha=0.7, pad=1))

        ax.set_title(Path(img_path).name[:25], fontsize=9)
        ax.axis('off')

    for ax in axes[len(samples):]:
        ax.axis('off')

    plt.suptitle(f'{split.upper()} Split — Sample Images with Annotations', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(WORK_DIR, f'samples_{split}.png'), dpi=150, bbox_inches='tight')
    plt.show()

# Show samples from each split
for split in ["train", "valid", "test"]:
    plot_sample_images(DATASET_DIR, split, CLASS_NAMES, CLASS_COLORS_HEX, n_samples=8)

## 5. Training — YOLOv11m Fine-Tuning

In [ ]:
# Load pretrained YOLOv11m
model = YOLO("yolo11m.pt")
print(f"Model: {model.model_name}")
print(f"Parameters: {sum(p.numel() for p in model.model.parameters()):,}")

In [ ]:
# Training configuration
TRAIN_CONFIG = {
    "data": yaml_path,
    "epochs": 80,
    "imgsz": 640,
    "batch": 32,
    "device": 0,
    "workers": 2,          # Kaggle has limited workers
    "seed": SEED,
    "project": WORK_DIR + "/runs",
    "name": "yolo11m_seatbelt_v1",
    "pretrained": True,
    "optimizer": "AdamW",
    "lr0": 0.01,
    "lrf": 0.01,
    "warmup_epochs": 3,
    "patience": 8,
    "save": True,
    "save_period": 10,
    "plots": True,
    "val": True,
    "cache": True,
    # Augmentation
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "degrees": 5,
    "translate": 0.1,
    "scale": 0.5,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "mixup": 0.1,
    "close_mosaic": 10,
}

print("Training config:")
for k, v in TRAIN_CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# Run training
results = model.train(**TRAIN_CONFIG)

## 6. Training Results & Metrics

In [ ]:
# Find best model
run_name = "yolo11m_seatbelt_v1"
results_dir = os.path.join(WORK_DIR, "runs", run_name)
best_model_path = os.path.join(results_dir, "weights", "best.pt")
best_model = YOLO(best_model_path)
print(f"Best model: {best_model_path}")
print(f"File size: {os.path.getsize(best_model_path) / 1e6:.1f} MB")

In [ ]:
# Display and save ALL ultralytics output images
from IPython.display import Image as IPImage, display
import shutil

presentation_dir = os.path.join(WORK_DIR, "presentation_outputs")
os.makedirs(presentation_dir, exist_ok=True)

ultralytics_images = [
    "results.png", "confusion_matrix.png", "confusion_matrix_normalized.png",
    "labels.jpg", "labels_correlogram.jpg",
    "P_curve.png", "R_curve.png", "PR_curve.png", "F1_curve.png",
    "BoxPR_curve.png",
]

for chart in ultralytics_images:
    path = os.path.join(results_dir, chart)
    if os.path.exists(path):
        print(f"\n{'='*60}")
        print(f"  {chart}")
        print(f"{'='*60}")
        display(IPImage(filename=path))
        shutil.copy2(path, os.path.join(presentation_dir, chart))
        print(f"  Saved to presentation_outputs/{chart}")

# Also check for any additional images
for ext in ["*.png", "*.jpg"]:
    for f in glob.glob(os.path.join(results_dir, ext)):
        fname = os.path.basename(f)
        if fname not in ultralytics_images:
            print(f"\n  Extra: {fname}")
            display(IPImage(filename=f))
            shutil.copy2(f, os.path.join(presentation_dir, f"extra_{fname}"))

print(f"\nAll presentation images saved to: {presentation_dir}")

In [ ]:
# Parse training results CSV for custom plots
csv_path = os.path.join(results_dir, "results.csv")
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    epochs = df['epoch']

    fig, axes = plt.subplots(2, 3, figsize=(20, 12))

    # Training losses
    for col, color, title in [
        ('train/box_loss', '#2196F3', 'Box Loss'),
        ('train/cls_loss', '#FF9800', 'Class Loss'),
        ('train/dfl_loss', '#9C27B0', 'DFL Loss'),
    ]:
        if col in df.columns:
            axes[0][0].plot(epochs, df[col], label=title, color=color)
    axes[0][0].set_title('Training Losses')
    axes[0][0].set_xlabel('Epoch')
    axes[0][0].legend()
    axes[0][0].grid(alpha=0.3)

    # Validation losses
    for col, color, title in [
        ('val/box_loss', '#2196F3', 'Box Loss'),
        ('val/cls_loss', '#FF9800', 'Class Loss'),
        ('val/dfl_loss', '#9C27B0', 'DFL Loss'),
    ]:
        if col in df.columns:
            axes[0][1].plot(epochs, df[col], label=title, color=color)
    axes[0][1].set_title('Validation Losses')
    axes[0][1].set_xlabel('Epoch')
    axes[0][1].legend()
    axes[0][1].grid(alpha=0.3)

    # mAP curves
    for col, color, label in [
        ('metrics/mAP50(B)', '#4CAF50', 'mAP@50'),
        ('metrics/mAP50-95(B)', '#F44336', 'mAP@50-95'),
    ]:
        if col in df.columns:
            axes[0][2].plot(epochs, df[col], label=label, color=color, linewidth=2)
    axes[0][2].set_title('mAP Scores')
    axes[0][2].set_xlabel('Epoch')
    axes[0][2].legend()
    axes[0][2].grid(alpha=0.3)

    # Precision / Recall
    for col, color, label in [
        ('metrics/precision(B)', '#2196F3', 'Precision'),
        ('metrics/recall(B)', '#FF9800', 'Recall'),
    ]:
        if col in df.columns:
            axes[1][0].plot(epochs, df[col], label=label, color=color, linewidth=2)
    axes[1][0].set_title('Precision & Recall')
    axes[1][0].set_xlabel('Epoch')
    axes[1][0].legend()
    axes[1][0].grid(alpha=0.3)

    # Learning rate
    lr_cols = [c for c in df.columns if 'lr' in c.lower()]
    for col in lr_cols:
        axes[1][1].plot(epochs, df[col], label=col)
    axes[1][1].set_title('Learning Rate Schedule')
    axes[1][1].set_xlabel('Epoch')
    axes[1][1].legend()
    axes[1][1].grid(alpha=0.3)

    # Final metrics summary
    final = df.iloc[-1]
    metric_names = ['metrics/precision(B)', 'metrics/recall(B)',
                   'metrics/mAP50(B)', 'metrics/mAP50-95(B)']
    metric_labels = ['Precision', 'Recall', 'mAP@50', 'mAP@50-95']
    vals = [final.get(m, 0) for m in metric_names]
    colors = ['#2196F3', '#FF9800', '#4CAF50', '#F44336']
    axes[1][2].barh(metric_labels, vals, color=colors)
    axes[1][2].set_xlim(0, 1)
    axes[1][2].set_title(f'Final Metrics (Epoch {int(final["epoch"])})')
    for i, v in enumerate(vals):
        axes[1][2].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=11)

    plt.suptitle('YOLOv11m Seatbelt Detection — Training Analysis', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(WORK_DIR, 'training_analysis.png'), dpi=150)
    plt.show()

    # Print final metrics
    print("\n" + "="*50)
    print("  FINAL TRAINING METRICS")
    print("="*50)
    for m, l in zip(metric_names, metric_labels):
        if m in df.columns:
            print(f"  {l:>15}: {df[m].iloc[-1]:.4f}")
    print(f"  {'Best Epoch':>15}: {df['metrics/mAP50-95(B)'].idxmax()}")
    print(f"  {'Best mAP@50-95':>15}: {df['metrics/mAP50-95(B)'].max():.4f}")

## 7. Per-Class Performance Analysis

In [ ]:
# Run validation on test set
val_results = best_model.val(data=yaml_path, split="test", verbose=True)

In [ ]:
# Per-class metrics visualization
per_class = val_results.box

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

metric_data = {}
for attr, title, ax in [
    ('p', 'Per-Class Precision', axes[0]),
    ('r', 'Per-Class Recall', axes[1]),
    ('ap50', 'Per-Class AP@50', axes[2]),
]:
    if hasattr(per_class, attr) and getattr(per_class, attr) is not None:
        values = getattr(per_class, attr)
        values = values.tolist() if hasattr(values, 'tolist') else values
        metric_data[attr] = values
        names = CLASS_NAMES[:len(values)]
        colors = ['#4CAF50' if v > 0.8 else '#FF9800' if v > 0.6 else '#F44336' for v in values]
        ax.barh(names, values, color=colors)
        ax.set_xlim(0, 1)
        ax.set_title(title)
        ax.axvline(0.8, color='gray', linestyle='--', alpha=0.5)
        for i, v in enumerate(values):
            ax.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)

plt.suptitle('Per-Class Performance on Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'per_class_metrics.png'), dpi=150)
plt.show()

# Print per-class table
print("\n" + "="*70)
print(f"  {'Class':>20} {'Precision':>10} {'Recall':>10} {'AP@50':>10} {'AP@50-95':>10}")
print("="*70)
p_values = metric_data.get('p', [])
r_values = metric_data.get('r', [])
ap_values = metric_data.get('ap50', [])
for i, name in enumerate(CLASS_NAMES[:len(p_values)]):
    p = p_values[i] if i < len(p_values) else 0
    r = r_values[i] if i < len(r_values) else 0
    ap50 = ap_values[i] if i < len(ap_values) else 0
    ap = per_class.ap.tolist()[i] if hasattr(per_class.ap, 'tolist') and i < len(per_class.ap) else 0
    flag = " WARNING" if p < 0.7 or r < 0.7 else ""
    print(f"  {name:>20} {p:>10.4f} {r:>10.4f} {ap50:>10.4f} {ap:>10.4f}{flag}")
print("="*70)

In [ ]:
# Class performance vs instance count correlation
fig, ax = plt.subplots(figsize=(10, 7))
test_counts = Counter()
lbl_dir = os.path.join(DATASET_DIR, "test", "labels")
if os.path.exists(lbl_dir):
    for lbl_file in glob.glob(os.path.join(lbl_dir, "*.txt")):
        with open(lbl_file) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cid = int(parts[0])
                    if cid < len(CLASS_NAMES):
                        test_counts[CLASS_NAMES[cid]] += 1

if 'ap50' in metric_data and test_counts:
    counts = [test_counts.get(n, 0) for n in CLASS_NAMES[:len(ap_values)]]
    aps = ap_values[:len(counts)]

    scatter = ax.scatter(counts, aps, c=CLASS_COLORS_HEX[:len(counts)],
                        s=200, alpha=0.7, edgecolors='black', zorder=5)
    for i, name in enumerate(CLASS_NAMES[:len(counts)]):
        ax.annotate(name, (counts[i], aps[i]), fontsize=9,
                   xytext=(8, 8), textcoords='offset points')

    ax.set_xlabel('Test Set Instance Count', fontsize=12)
    ax.set_ylabel('AP@50', fontsize=12)
    ax.set_title('Does More Data = Better Accuracy?', fontsize=14)
    ax.axhline(0.8, color='gray', linestyle='--', alpha=0.3)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(WORK_DIR, 'data_vs_accuracy.png'), dpi=150)
    plt.show()

## 8. Prediction Samples on Test Set

In [ ]:
# Run predictions on test set and visualize
test_img_dir = os.path.join(DATASET_DIR, "test", "images")
test_imgs = sorted(glob.glob(os.path.join(test_img_dir, "*.*")))[:12]

if test_imgs:
    preds = best_model.predict(test_imgs, conf=0.25, save=False, verbose=False)

    fig, axes = plt.subplots(3, 4, figsize=(20, 15))
    axes = axes.flatten()

    for idx, (img_path, result) in enumerate(zip(test_imgs, preds)):
        img = Image.open(img_path)
        axes[idx].imshow(np.array(img))

        if result.boxes is not None:
            for box in result.boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                conf = box.conf[0].cpu().item()
                cls_id = int(box.cls[0].cpu().item())
                label = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else str(cls_id)
                color = CLASS_COLORS_HEX[cls_id] if cls_id < len(CLASS_COLORS_HEX) else 'lime'

                rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                        linewidth=2, edgecolor=color, facecolor='none')
                axes[idx].add_patch(rect)
                axes[idx].text(x1, y1 - 3, f"{label} {conf:.2f}", fontsize=7,
                              color='white', bbox=dict(facecolor=color, alpha=0.7, pad=1))

        axes[idx].set_title(Path(img_path).name[:20], fontsize=8)
        axes[idx].axis('off')

    for ax in axes[len(test_imgs):]:
        ax.axis('off')

    plt.suptitle('Test Set Predictions (Colored by Class)', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(WORK_DIR, 'test_predictions.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 9. Export & Save

In [ ]:
# Copy best weights
final_path = os.path.join(WORK_DIR, "seatbelt_v1.pt")
shutil.copy2(best_model_path, final_path)
print(f"Model saved to: {final_path}")
print(f"File size: {os.path.getsize(final_path) / 1e6:.1f} MB")
print("\nDownload and place in RADAR/models/seatbelt_v1.pt")

In [ ]:
# Zip ALL outputs for easy download
output_dir = os.path.join(WORK_DIR, "seatbelt_v1_all_outputs")
os.makedirs(output_dir, exist_ok=True)

# Best model
if os.path.exists(best_model_path):
    shutil.copy2(best_model_path, os.path.join(output_dir, "seatbelt_v1.pt"))
    print(f"Model: {os.path.getsize(best_model_path)/1e6:.1f} MB")

# Presentation images
pres_dir = os.path.join(WORK_DIR, "presentation_outputs")
if os.path.exists(pres_dir):
    for f in glob.glob(os.path.join(pres_dir, "*.*")):
        shutil.copy2(f, output_dir)

# Custom analysis plots
for f in glob.glob(os.path.join(WORK_DIR, "*.png")):
    shutil.copy2(f, output_dir)

# Training CSV results
csv_src = os.path.join(results_dir, "results.csv")
if os.path.exists(csv_src):
    shutil.copy2(csv_src, output_dir)

# Args.yaml
args_src = os.path.join(results_dir, "args.yaml")
if os.path.exists(args_src):
    shutil.copy2(args_src, output_dir)

# Zip it all
shutil.make_archive(output_dir, 'zip', WORK_DIR, "seatbelt_v1_all_outputs")
print(f"\nAll outputs zipped to: {output_dir}.zip")
print(f"Files included:")
for f in sorted(os.listdir(output_dir)):
    size = os.path.getsize(os.path.join(output_dir, f))
    print(f"  {f} ({size/1e3:.0f} KB)" if size < 1e6 else f"  {f} ({size/1e6:.1f} MB)")

In [ ]:
# Zip the entire runs folder (checkpoints, weights, all plots)
runs_zip = os.path.join(WORK_DIR, "runs_backup")
shutil.make_archive(runs_zip, 'zip', WORK_DIR, "runs")
print(f"Runs folder zipped to: {runs_zip}.zip")
print(f"Size: {os.path.getsize(runs_zip + '.zip') / 1e6:.1f} MB")